# MedSAM Baseline Test - Medical Image Segmentation Evaluation

This notebook provides comprehensive baseline evaluation of Medical SAM (MedSAM) on multiple medical image segmentation datasets including:
- 🧠 **MRI**: Hippocampus segmentation (brain structures)
- 🫁 **CT**: Spleen segmentation (abdominal organ)  
- 🔬 **Ultrasound**: Breast tumor segmentation

**Prerequisites:**
- RL-CC-SAM environment activated with 'llms' virtual environment
- Required dependencies from requirements.txt
- GPU runtime recommended (Runtime > Change runtime type > GPU)

**References:**
- MedSAM paper: https://arxiv.org/abs/2304.12306
- Original SAM: segment-anything/README.md
- Medical Decathlon datasets: http://medicaldecathlon.com/

**Estimated Runtime:** 45-90 minutes (depending on dataset size and hardware)

## 🔧 1. Environment Setup and Configuration

Following the established patterns from setup_colab.ipynb and compliance with .cursor/rules

In [ ]:
# Environment setup and Colab detection
# Ref: setup_colab.ipynb for Colab integration patterns
import sys
import os
from pathlib import Path
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("🌐 Running in Google Colab")
    from google.colab import drive
    drive.mount('/content/drive')

    # Set up paths for Colab - following setup_colab.ipynb patterns
    DRIVE_ROOT = '/content/drive/MyDrive'
    PROJECT_ROOT = f'{DRIVE_ROOT}/RL-CC-SAM'

    # Change to project directory
    os.chdir(PROJECT_ROOT)
    sys.path.append(PROJECT_ROOT)

    print(f"📁 Working directory: {os.getcwd()}")
else:
    print("💻 Running locally")
    # Assume notebook is in notebooks/ folder
    PROJECT_ROOT = Path.cwd().parent
    os.chdir(PROJECT_ROOT)

    print(f"📁 Working directory: {PROJECT_ROOT}")

# Set up directories following project structure
DATASETS_DIR = Path(PROJECT_ROOT) / "datasets"
PRETRAINED_DIR = Path(PROJECT_ROOT) / "pretrained"
SEGMENT_ANYTHING_DIR = Path(PROJECT_ROOT) / "segment-anything"

# Create directories
for dir_path in [DATASETS_DIR, PRETRAINED_DIR]:
    dir_path.mkdir(exist_ok=True)

print(f"📊 Datasets directory: {DATASETS_DIR}")
print(f"🧠 Pretrained models directory: {PRETRAINED_DIR}")
print(f"🔧 Segment Anything submodule: {SEGMENT_ANYTHING_DIR}")

# Verify submodules are available (following .cursor/rules/submodule-handling.mdc)
if not SEGMENT_ANYTHING_DIR.exists():
    print("❌ Segment Anything submodule not found!")
    print("💡 Run: git submodule update --init --recursive")
    print("💡 Or use setup_colab.ipynb for initial setup")
else:
    print("✅ Segment Anything submodule found")

print("✅ Environment setup complete")


## 📚 2. Library Imports and Dependencies

All dependencies are managed through requirements.txt (following .cursor/rules/python-jupyter.mdc)

In [ ]:
# Import essential libraries
# All dependencies should be installed via requirements.txt

# Core libraries
import torch
import numpy as np
import matplotlib.pyplot as plt
import cv2
import json
import math
import glob
from tqdm.auto import tqdm

# Medical imaging
import nibabel as nib
from scipy.ndimage import distance_transform_edt

# SAM - using existing submodule (following .cursor/rules/submodule-handling.mdc)
sys.path.insert(0, str(SEGMENT_ANYTHING_DIR))
from segment_anything import sam_model_registry, SamPredictor

# Image processing
from skimage import io, transform
import torch.nn.functional as F

# Device handling following .cursor/rules/pytorch-devices.mdc
def get_device():
    """Get optimal device following PyTorch device handling rules."""
    if torch.backends.mps.is_available():
        return torch.device("mps")  # Apple Silicon
    elif torch.cuda.is_available():
        return torch.device("cuda")  # NVIDIA GPU
    else:
        return torch.device("cpu")   # CPU fallback

device = get_device()
print(f"🚀 Using device: {device}")

# Set style for plots
plt.style.use('seaborn-v0_8' if 'seaborn-v0_8' in plt.style.available else 'default')
print("✅ Libraries imported successfully")


## 📊 3. Evaluation Metrics and Helper Functions


In [ ]:
# Define comprehensive evaluation metrics for medical image segmentation
def compute_dice(pred_mask: np.ndarray, true_mask: np.ndarray) -> float:
    """
    Calculate Dice Coefficient (F1 score) between two binary masks.

    Args:
        pred_mask: Predicted binary mask
        true_mask: Ground truth binary mask

    Returns:
        float: Dice coefficient [0, 1], where 1 is perfect overlap
    """
    intersection = np.logical_and(pred_mask, true_mask).sum()
    size_pred = pred_mask.sum()
    size_true = true_mask.sum()

    if size_pred + size_true == 0:
        return 1.0  # Both masks are empty - perfect agreement

    return 2.0 * intersection / (size_pred + size_true + 1e-8)

def compute_iou(pred_mask: np.ndarray, true_mask: np.ndarray) -> float:
    """
    Calculate Intersection over Union (IoU) between two binary masks.

    Args:
        pred_mask: Predicted binary mask
        true_mask: Ground truth binary mask

    Returns:
        float: IoU score [0, 1], where 1 is perfect overlap
    """
    intersection = np.logical_and(pred_mask, true_mask).sum()
    union = np.logical_or(pred_mask, true_mask).sum()

    if union == 0:
        return 1.0  # Both masks are empty

    return intersection / (union + 1e-8)

def compute_hausdorff(pred_mask: np.ndarray, true_mask: np.ndarray) -> float:
    """
    Calculate Hausdorff distance between two binary masks.

    Args:
        pred_mask: Predicted binary mask
        true_mask: Ground truth binary mask

    Returns:
        float: Hausdorff distance in pixels (lower is better)
    """
    if pred_mask.sum() == 0 or true_mask.sum() == 0:
        return math.inf  # Cannot compute distance if either mask is empty

    # Calculate distance transforms
    dt_true = distance_transform_edt(~true_mask.astype(bool))
    dt_pred = distance_transform_edt(~pred_mask.astype(bool))

    # Calculate directed Hausdorff distances
    hd1 = dt_pred[true_mask.astype(bool)].max()  # GT boundary to pred region
    hd2 = dt_true[pred_mask.astype(bool)].max()  # Pred boundary to GT region

    return max(hd1, hd2)

def compute_sensitivity(pred_mask: np.ndarray, true_mask: np.ndarray) -> float:
    """Calculate sensitivity (recall/true positive rate)."""
    tp = np.logical_and(pred_mask, true_mask).sum()
    fn = np.logical_and(~pred_mask, true_mask).sum()

    if tp + fn == 0:
        return 1.0  # No positive cases

    return tp / (tp + fn)

def compute_specificity(pred_mask: np.ndarray, true_mask: np.ndarray) -> float:
    """Calculate specificity (true negative rate)."""
    tn = np.logical_and(~pred_mask, ~true_mask).sum()
    fp = np.logical_and(pred_mask, ~true_mask).sum()

    if tn + fp == 0:
        return 1.0  # No negative cases

    return tn / (tn + fp)

def evaluate_segmentation(pred_mask: np.ndarray, true_mask: np.ndarray) -> dict:
    """
    Comprehensive evaluation of segmentation performance.

    Returns:
        dict: Dictionary containing all evaluation metrics
    """
    return {
        'dice': compute_dice(pred_mask, true_mask),
        'iou': compute_iou(pred_mask, true_mask),
        'hausdorff': compute_hausdorff(pred_mask, true_mask),
        'sensitivity': compute_sensitivity(pred_mask, true_mask),
        'specificity': compute_specificity(pred_mask, true_mask)
    }

# Visualization functions
def show_mask(mask, ax, random_color=False, alpha=0.6):
    """Display segmentation mask overlay."""
    if random_color:
        color = np.concatenate([np.random.random(3), np.array([alpha])], axis=0)
    else:
        color = np.array([251/255, 252/255, 30/255, alpha])  # Yellow

    h, w = mask.shape[-2:]
    mask_image = mask.reshape(h, w, 1) * color.reshape(1, 1, -1)
    ax.imshow(mask_image)

def show_box(box, ax, color='blue'):
    """Draw bounding box on matplotlib axis."""
    x0, y0 = box[0], box[1]
    w, h = box[2] - box[0], box[3] - box[1]
    ax.add_patch(plt.Rectangle((x0, y0), w, h, edgecolor=color, facecolor=(0,0,0,0), lw=2))

print("✅ Evaluation metrics and helper functions defined")
print("📊 Available metrics: Dice, IoU, Hausdorff Distance, Sensitivity, Specificity")
############################
# import numpy.linalg as LA

# def hausdorff_distance_95(pred: np.ndarray, true: np.ndarray, spacing=(1.0,1.0,1.0)):
#     """
#     计算95% Hausdorff距离（基于表面点集）。
#     spacing用于将像素距离转换为实际单位（例如mm），对于2D图像可忽略Z轴spacing。
#     """
#     # 提取边界点坐标
#     import numpy as np
#     from scipy.ndimage import binary_erosion
#     pred = pred.astype(bool)
#     true = true.astype(bool)
#     # 若无前景，返回最大距离0或图像对角线
#     if pred.sum()==0 or true.sum()==0:
#         return np.nan if pred.sum()==0 and true.sum()==0 else float('inf')
#     pred_border = np.logical_xor(pred, binary_erosion(pred))
#     true_border = np.logical_xor(true, binary_erosion(true))
#     pred_pts = np.vstack(np.nonzero(pred_border)).T * spacing  # N x dim
#     true_pts = np.vstack(np.nonzero(true_border)).T * spacing
#     # 计算两集合的距离矩阵的min距离
#     from scipy.spatial.distance import cdist
#     dists = cdist(pred_pts, true_pts)
#     # 双向Hausdorff距离集合
#     min_pred_to_true = dists.min(axis=1)
#     min_true_to_pred = dists.min(axis=0)
#     all_distances = np.concatenate([min_pred_to_true, min_true_to_pred])
#     # 95百分位
#     return np.percentile(all_distances, 95)

## 🧠 4. MedSAM Model Setup and Loading

Following best practices for model management in the pretrained/ directory

In [ ]:
# Load MedSAM model (assumes already downloaded via setup_colab.ipynb)
# Following .cursor/rules: Downloads handled in setup_colab.ipynb

def load_medsam_model():
    """
    Load MedSAM model from pretrained directory.

    Prerequisites:
    - Run setup_colab.ipynb first to download the model
    - Model should be at: pretrained/medsam_vit_b.pth
    """

    model_filename = "medsam_vit_b.pth"
    model_path = PRETRAINED_DIR / model_filename

    # Check if model exists
    if not model_path.exists():
        print(f"❌ MedSAM model not found at: {model_path}")
        print("💡 Please run setup_colab.ipynb first to download the model")
        print("💡 The setup notebook will download all required models and datasets")
        raise FileNotFoundError(f"MedSAM model not found. Run setup_colab.ipynb first.")

    try:
        print(f"🔄 Loading MedSAM model from: {model_path}")

        # Load using SAM architecture with MedSAM weights
        sam_model = sam_model_registry["vit_b"](checkpoint=str(model_path))
        sam_model = sam_model.to(device)
        sam_model.eval()

        # Create predictor
        predictor = SamPredictor(sam_model)

        print(f"✅ MedSAM model loaded successfully")
        print(f"   Model size: {model_path.stat().st_size / (1024**3):.1f} GB")
        print(f"   Model parameters: {sum(p.numel() for p in sam_model.parameters()):,}")
        print(f"   Device: {device}")

        return sam_model, predictor

    except Exception as e:
        print(f"❌ Error loading MedSAM model: {e}")
        print("💡 The model file might be corrupted. Re-run setup_colab.ipynb")
        raise

# Load the MedSAM model
medsam_model, predictor = load_medsam_model()


In [ ]:
MEDICAL_DECATHLON_DIR = DATASETS_DIR / "medical_decathlon"
Task04_Hippocampus_dir = MEDICAL_DECATHLON_DIR / "Task04_Hippocampus"
Task09_Spleen_dir = MEDICAL_DECATHLON_DIR / "Task09_Spleen"
BUSI_Dataset_dir = DATASETS_DIR / "BUSI_Dataset"

接下来读取MRI体积数据和对应的标签。在评估中，我们将使用训练集作为测试图像（MedSAM并未在该特定数据上训练过，因此训练集可以用作评估基线）。我们将利用Nibabel库读取NIfTI格式的体数据，并提取其中的2D切片进行推理：

In [ ]:
## 代码遍历了MRI体数据的每个切片，仅保留含有海马标注的切片，以减少不必要的计算。每个切片被归一化为三通道8位图像（MedSAM需要RGB图像输入）。mri_slices列表将用于模型推理，mri_slice_masks为对应的真值掩码。
import nibabel as nib
import numpy as np

# 获取Hippocampus训练集文件列表
imagesTr = !ls -1 {Task04_Hippocampus_dir}/imagesTr
labelsTr = !ls -1 {Task04_Hippocampus_dir}/labelsTr
image_files = sorted([f for f in imagesTr])
label_files = sorted([f for f in labelsTr])

print(f"共有 {len(image_files)} 副MRI体积用于评估.")

# 准备存储MRI数据集的测试切片和标签
mri_slices = []      # 将保存二维切片图像 (numpy数组)
mri_slice_masks = [] # 将保存对应的二维GT掩码
mri_slice_labels = []# 保存掩码对应的解剖结构标签（1=左海马, 2=右海马）
mri_slice_summaries = []

for img_file, lbl_file in zip(image_files, label_files):
    img_path = f"{Task04_Hippocampus_dir}/imagesTr/{img_file}"
    lbl_path = f"{Task04_Hippocampus_dir}/labelsTr/{lbl_file}"
    # 载入NIfTI体数据
    img_nii = nib.load(img_path)
    seg_nii = nib.load(lbl_path)
    img_data = img_nii.get_fdata()
    seg_data = seg_nii.get_fdata()
    volume = img_data.astype(np.float32)  # 3D影像数据
    seg_volume = seg_data.astype(np.uint8)

    # # 归一化强度（Z-score）
    # mask = volume > 0  # MRI可能有背景0
    # voxels = volume[mask]
    # volume[mask] = (voxels - voxels.mean()) / (voxels.std() + 1e-8)

    # 遍历每个切片（以轴2为切片方向，即轴向切片）
    num_slices = img_data.shape[2]
    for z_index in range(num_slices):
        # 取某一轴向切片
        slice_img = volume[:, :, z_index]
        slice_mask = seg_volume[:, :, z_index]
        # 如果该切片存在任何一个海马结构的标注，则纳入评估
        if np.any(slice_mask > 0):
            print(f"\rimg_file:'{img_file}', lbl_file:{lbl_file}, |volume|: {volume.shape}, |seg|: {seg_volume.shape}, slice: {z_index}/{num_slices}, Slice shape: {slice_img.shape}, Mask unique labels: {np.unique(slice_mask)}", end='', flush=True)
            mri_slices.append(slice_img)
            mri_slice_masks.append(slice_mask)  # 多类别标签（值0,1,2）
            mri_slice_labels.append(lbl_file)  # 记录该切片所属体数据文件
            mri_slice_summaries.append((z_index, num_slices, slice_mask.sum() / slice_img.shape[0] / slice_img.shape[1]))


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.figure import Figure
from matplotlib.gridspec import GridSpec
from matplotlib.axes import Axes

def visualize_seg(fig: Figure, axs: Axes, label2box: dict, label2mask: dict, img_rgb: np.ndarray, true_mask: np.ndarray):
  # fig.clear()
  for structure_idx, (label_val, pred_mask) in enumerate(label2mask.items()):
        # Get box for this structure
        input_box = label2box[label_val]
        if input_box is None:
            continue
        x_min, y_min, x_max, y_max = input_box

        row = structure_idx
        # 1. Original image with bounding box
        ax1 = axs[row, 0]
        ax1.clear()
        ax1.imshow(img_rgb[:,:,0], cmap='gray')
        rect = plt.Rectangle((x_min, y_min), x_max-x_min, y_max-y_min,
                            fill=False, edgecolor='red', linewidth=2)
        ax1.add_patch(rect)
        ax1.set_title(f'Input Box - Structure {label_val}')
        ax1.axis('off')

        # 2. True mask for this structure
        ax2 = axs[row, 1]
        ax2.clear()
        structure_true_mask = (true_mask == label_val).astype(np.uint8)
        ax2.imshow(img_rgb[:,:,0], cmap='gray')
        ax2.imshow(np.ma.masked_where(structure_true_mask==0, structure_true_mask),
                  cmap='spring', alpha=0.7)
        ax2.set_title(f'True Mask - Structure {label_val}')
        ax2.axis('off')

        # 3. Predicted mask for this structure
        ax3 = axs[row, 2]
        ax3.clear()
        ax3.imshow(img_rgb[:,:,0], cmap='gray')
        ax3.imshow(np.ma.masked_where(pred_mask==0, pred_mask),
                  cmap='autumn', alpha=0.7)
        ax3.set_title(f'Pred Mask - Structure {label_val}')
        ax3.axis('off')

        # 4. Overlay comparison
        ax4 = axs[row, 3]
        ax4.clear()
        ax4.imshow(img_rgb[:,:,0], cmap='gray')
        # True mask in green, pred mask in red
        ax4.imshow(np.ma.masked_where(structure_true_mask==0, structure_true_mask),
                  cmap='Greens', alpha=0.5)
        ax4.imshow(np.ma.masked_where(pred_mask==0, pred_mask),
                  cmap='Reds', alpha=0.5)
        ax4.set_title(f'Overlay - Structure {label_val}')
        ax4.axis('off')

  # Combined visualization (third row)
  # 1. All bounding boxes
  ax_combined1 = axs[2, 0]
  ax_combined1.clear()
  ax_combined1.imshow(img_rgb[:,:,0], cmap='gray')
  colors = ['red', 'blue']
  for i, (label_val, input_box) in enumerate(label2box.items()):
      x_min, y_min, x_max, y_max = input_box
      rect = plt.Rectangle((x_min, y_min), x_max-x_min, y_max-y_min,
                          fill=False, edgecolor=colors[i % len(colors)], linewidth=2)
      ax_combined1.add_patch(rect)
  ax_combined1.set_title('All Input Boxes')
  ax_combined1.axis('off')

  # 2. Combined true mask
  ax_combined2 = axs[2, 1]
  ax_combined2.clear()
  ax_combined2.imshow(img_rgb[:,:,0], cmap='gray')
  ax_combined2.imshow(np.ma.masked_where((true_mask > 0)==0, (true_mask > 0)),
                      cmap='spring', alpha=0.7)
  ax_combined2.set_title('Combined True Mask')
  ax_combined2.axis('off')

  # 3. Combined predicted mask
  ax_combined3 = axs[2, 2]
  ax_combined3.clear()
  ax_combined3.imshow(img_rgb[:,:,0], cmap='gray')
  ax_combined3.imshow(np.ma.masked_where(pred_mask_total==0, pred_mask_total),
                      cmap='autumn', alpha=0.7)
  ax_combined3.set_title('Combined Pred Mask')
  ax_combined3.axis('off')

  # 4. Final comparison
  ax_combined4 = axs[2, 3]
  ax_combined4.clear()
  ax_combined4.imshow(img_rgb[:,:,0], cmap='gray')
  ax_combined4.imshow(np.ma.masked_where((true_mask > 0)==0, (true_mask > 0)),
                      cmap='Greens', alpha=0.5)
  ax_combined4.imshow(np.ma.masked_where(pred_mask_total==0, pred_mask_total),
                      cmap='Reds', alpha=0.5)
  ax_combined4.set_title('Final Overlay')
  ax_combined4.axis('off')

  # # Update figure title with metrics - highlight poor performance
  # fig.suptitle(f'Slice {idx+1}/{len(mri_slices)}\n'
  #             f'Dice: {dice:.3f}, IoU: {iou:.3f}, HD: {hd:.1f}px',
  #             fontsize=14, y=0.95, color='red')

  fig.canvas.draw()
  # fig.canvas.draw_idle()
  fig.canvas.flush_events()
  # plt.show()
  # plt.pause(1.0)

    # # Wait for user input to continue
  # try:
  #     user_input = input(f"\n👁️  Visualization #{visualization_count} shown. Press Enter to continue (or 'q' to stop visualizations): ")
  #     if user_input.lower() == 'q':
  #         print("🛑 Stopping visualizations. Continuing evaluation without display...")
  #         if fig is not None:
  #             plt.close(fig)
  #             fig = None
  #         break
  # except (KeyboardInterrupt, EOFError):
  #     print("\n🛑 Visualization interrupted. Continuing evaluation...")
  #     if fig is not None:
  #         plt.close(fig)
  #         fig = None
  #     break

all_labels = [1, 2]

def medsam_segment(slice_img: np.ndarray, slice_mask: np.ndarray, predictor: SamPredictor):
    ###slice_clip = np.clip(slice_img, -1000, 1000)
    mn, mx = slice_img.min(), slice_img.max()
    slice_img_norm = ((slice_img - mn) / (mx - mn + 1e-8) * 255.0).astype(np.uint8)
    slice_img_resized = cv2.resize(slice_img_norm, (1024, 1024), interpolation=cv2.INTER_CUBIC)
    img_rgb = np.stack([slice_img_resized]*3, axis=-1)

    true_mask = cv2.resize(slice_mask.astype(np.uint8)*127, (1024, 1024), interpolation=cv2.INTER_NEAREST)
    true_mask = true_mask.astype(np.uint8) // 127 ## OR:
    ###true_mask = (true_mask > 127).astype(np.uint8)

    predictor.set_image(img_rgb)  # 计算图像嵌入
    pred_mask_total = np.zeros(true_mask.shape, dtype=bool)

    # 对每个结构分别预测 (标签1和2分别对应两个海马)
    label2mask = {}
    label2box = {}
    for structure_idx, label_val in enumerate(all_labels):
        # 跳过不存在的结构
        if np.sum(true_mask == label_val) == 0:
            continue

        # 计算该结构的边界框
        ys, xs = np.where(true_mask == label_val)
        y_min, y_max = ys.min(), ys.max()
        x_min, x_max = xs.min(), xs.max()
        input_box = np.array([x_min, y_min, x_max, y_max])
        label2box[label_val] = input_box

        # 使用边界框提示进行预测
        masks, scores, _ = predictor.predict(box=input_box[None, :], point_coords=None, point_labels=None, multimask_output=False)
        pred_mask = masks[0]  # 输出掩码
        label2mask[label_val] = pred_mask

        # 将该结构的预测掩码累加到总掩码
        pred_mask_total = np.logical_or(pred_mask_total, pred_mask)

    return label2mask, label2box, pred_mask_total, true_mask, img_rgb

In [ ]:
# 评估MRI (Hippocampus) 数据集
dice_list_hc = []
iou_list_hc = []
hd_list_hc = []

good_slices = []

for idx, (slice_img, slice_mask, slice_label, slice_info) in enumerate(zip(mri_slices, mri_slice_masks, mri_slice_labels, mri_slice_summaries)):
    # if idx > 20:
    #    break
    label2mask, label2box, pred_mask_total, true_mask, img_rgb = medsam_segment(slice_img, slice_mask, predictor)
    combined_true_mask = (true_mask > 0)

    if slice_info[2] > 0.12 and len(label2box) > 1:
      good_slices.append(idx)

    # 计算评价指标
    dice = compute_dice(pred_mask_total, combined_true_mask)
    iou = compute_iou(pred_mask_total, combined_true_mask)
    hd = compute_hausdorff(pred_mask_total, combined_true_mask)
    dice_list_hc.append(dice)
    iou_list_hc.append(iou)
    hd_list_hc.append(hd)
    # 打印处理进度
    print(f"\rProcessing slice {idx+1}/{len(mri_slices)}: {slice_label}, "
          f"Dice: {dice:.3f}, IoU: {iou:.3f}, HD: {hd:.1f}px", end='', flush=True)

In [ ]:
dice_list_hc = np.array(dice_list_hc)
iou_list_hc = np.array(iou_list_hc)
hd_list_hc = np.array([d for d in hd_list_hc if math.isfinite(d)])

# 计算平均指标
dice_mean_hc = np.mean(dice_list_hc)
iou_mean_hc = np.mean(iou_list_hc)
hd_mean_hc = np.mean(hd_list_hc)

print(f"\n\n📊 Final Results Summary:")
print(f"   Total slices processed: {len(mri_slices)}")
# print(f"   Low DICE visualizations shown: {visualization_count}")
print(f"   Slices with DICE < 0.5: {sum(1 for d in dice_list_hc if d < 0.5)}")
print(f"   Hippocampus MRI数据集: 平均Dice = {dice_mean_hc:.4f}, 平均IoU = {iou_mean_hc:.4f}, 平均Hausdorff距离 = {hd_mean_hc:.2f} pixel")

good_dice_mean_hc = np.mean(dice_list_hc[good_slices])
good_iou_mean_hc = np.mean(iou_list_hc[good_slices])
good_hd_mean_hc = np.mean(hd_list_hc[good_slices])
print(f"   In slices with maskedRatio > 0.12 and both labels existing:")
print(f"   平均Dice = {good_dice_mean_hc:.4f}, 平均IoU = {good_iou_mean_hc:.4f}, 平均Hausdorff距离 = {good_hd_mean_hc:.2f} pixel")

In [ ]:
# Clean up
fig = None
plt.ioff()  # Turn off interactive mode
# plt.show()
# Set up visualization
plt.ion()  # Turn on interactive mode
# fig = plt.figure(figsize=(16, 10))
# # Create subplots for visualization
# gs = fig.add_gridspec(3, 4, hspace=0.3, wspace=0.2)
fig, axs = plt.subplots(3, 4)

idx = good_slices[np.random.randint(0, len(good_slices))]
slice_img, slice_mask, slice_label, slice_info = mri_slices[idx], mri_slice_masks[idx], mri_slice_labels[idx], mri_slice_summaries[idx]
label2mask, label2box, pred_mask_total, true_mask, img_rgb = medsam_segment(slice_img, slice_mask, predictor)
# Visualize each structure separately
visualize_seg(fig, axs, label2box, label2mask, img_rgb, true_mask)

# Update figure title with metrics - highlight poor performance
dice, iou, hd = dice_list_hc[idx], iou_list_hc[idx], hd_list_hc[idx]
fig.suptitle(f'Slice {idx+1}/{len(mri_slices)}\n'
            f'Dice: {dice:.3f}, IoU: {iou:.3f}, HD: {hd:.1f}px',
            fontsize=14, y=0.95, color='red')
# Clean up
plt.ioff()  # Turn off interactive mode
plt.tight_layout()
plt.show()
plt.close(fig)
fig = None

下载CT数据集（Spleen）
Spleen数据集来自Medical Segmentation Decathlon Task09
academictorrents.com
。该数据集提供腹部CT体数据及脾脏的分割标签（二值掩码，1表示脾脏）。我们同样使用AWS接口下载Task09_Spleen数据集，并解压：

In [ ]:
# # 下载Decathlon Task09 (Spleen) 数据集 (~1.5GB)
# !aws s3 cp --no-sign-request s3://msd-for-monai/Task09_Spleen.tar .
# !tar -xf Task09_Spleen.tar
# !rm Task09_Spleen.tar

# 列出Spleen数据集目录结构
!find {Task09_Spleen_dir} -maxdepth 1 -type d -printf '%P\n'


下载完成后，我们读取CT体数据和标签，并提取有脾脏的切片进行评估：

In [ ]:
# 获取Spleen数据集文件列表
ct_image_files = !ls -1 {Task09_Spleen_dir}/imagesTr
ct_label_files = !ls -1 {Task09_Spleen_dir}/labelsTr
ct_image_files = sorted([f for f in ct_image_files])
ct_label_files = sorted([f for f in ct_label_files])
print(f"共有 {len(ct_image_files)} 副CT体积用于评估.")

ct_slices = []
ct_slice_masks = []

for img_file, lbl_file in zip(ct_image_files, ct_label_files):
    print(f"\rimg_file:'{img_file}', lbl_file:{lbl_file}", end='', flush=True)
    img_path = f"{Task09_Spleen_dir}/imagesTr/{img_file}"
    lbl_path = f"{Task09_Spleen_dir}/labelsTr/{lbl_file}"
    img_nii = nib.load(img_path)
    lbl_nii = nib.load(lbl_path)
    img_data = img_nii.get_fdata().astype(np.float32)
    lbl_data = lbl_nii.get_fdata().astype(np.uint8)
    # 遍历轴向切片
    num_slices = img_data.shape[0]
    for k in range(num_slices):
        slice_img = img_data[k, :, :]
        slice_lbl = lbl_data[k, :, :]
        if np.any(slice_lbl == 1):  # 若该切片含有脾脏
            # 将CT切片灰度值裁剪到 [-1000, 1000] HU 范围，并归一化到0-255
            slice_clip = np.clip(slice_img, -1000, 1000)
            # mn, mx = slice_clip.min(), slice_clip.max()
            # slice_img_norm = ((slice_clip - mn) / (mx - mn + 1e-8) * 255.0).astype(np.uint8)
            # slice_img_rgb = np.stack([slice_img_norm]*3, axis=-1)
            ct_slices.append(slice_clip)
            ct_slice_masks.append(slice_lbl)  # 二值掩码（0背景，1脾脏）


在以上代码中，对于CT强度值，我们对每个切片裁剪在[-1000,1000]范围以排除空气和高密度异常值，然后进行min-max归一化。这样可以保留软组织和脏器的对比。只提取包含脾脏的切片到ct_slices列表中。

In [ ]:
dice_list_spleen = []
iou_list_spleen = []
hd_list_spleen = []

for idx, (slice_img, slice_mask) in enumerate(zip(ct_slices, ct_slice_masks)):
    # if idx > 20:
    #    break
    label2mask, label2box, pred_mask_total, true_mask, img_rgb = medsam_segment(slice_img, slice_mask, predictor)
    # combined_true_mask = (true_mask > 0)
    combined_true_mask = (true_mask == 1)

    # 计算评价指标
    dice = compute_dice(pred_mask_total, combined_true_mask)
    iou = compute_iou(pred_mask_total, combined_true_mask)
    hd = compute_hausdorff(pred_mask_total, combined_true_mask)
    dice_list_spleen.append(dice)
    iou_list_spleen.append(iou)
    hd_list_spleen.append(hd)
    # 打印处理进度
    print(f"\rProcessing CT slice {idx+1}/{len(ct_slices)}: "
          f"Dice: {dice:.3f}, IoU: {iou:.3f}, HD: {hd:.1f}px", end='', flush=True)

In [ ]:
# 计算平均指标
dice_mean_spleen = np.mean(dice_list_spleen)
iou_mean_spleen = np.mean(iou_list_spleen)
hd_mean_spleen = np.mean([d for d in hd_list_spleen if math.isfinite(d)])
print(f"\n\n📊 Final Results Summary:")
print(f"   Total slices processed: {len(ct_slices)}")
print(f"   Slices with DICE < 0.5: {sum(1 for d in dice_list_spleen if d < 0.5)}")
print(f"Spleen CT数据集: 平均Dice = {dice_mean_spleen:.4f}, 平均IoU = {iou_mean_spleen:.4f}, 平均Hausdorff距离 = {hd_mean_spleen:.2f} pixel")

In [ ]:
if fig is not None:
  fig.clear()
  plt.close(fig)
  fig = None
plt.ion()  # Turn on interactive mode
fig, axs = plt.subplots(3, 4)

idx = np.random.randint(0, len(ct_slices))
slice_img, slice_mask = ct_slices[idx], ct_slice_masks[idx]
label2mask, label2box, pred_mask_total, true_mask, img_rgb = medsam_segment(slice_img, slice_mask, predictor)
visualize_seg(fig, axs, label2box, label2mask, img_rgb, true_mask)

# Update figure title with metrics - highlight poor performance
dice, iou, hd = dice_list_spleen[idx], iou_list_spleen[idx], hd_list_spleen[idx]
fig.suptitle(f'Slice {idx+1}/{len(mri_slices)}\n'
            f'Dice: {dice:.3f}, IoU: {iou:.3f}, HD: {hd:.1f}px',
            fontsize=14, y=0.95, color='red')
# Clean up
plt.ioff()  # Turn off interactive mode
plt.tight_layout()
plt.show()
plt.close(fig)
fig = None

3. 下载超声数据集（BUSB/BUSI）
BUSB（Breast Ultrasound Images Dataset）数据集包含780张超声图像及其分割掩码，包括正常、良性和恶性三类病例
academictorrents.com
datasetninja.com
。其中正常类没有肿块（无分割掩码），良性和恶性病例有肿瘤掩码。我们下载该数据集的公开发布版本，并解压得到图像及标注。这里我们假设数据集以zip文件提供，并采用bash命令获取：

📓 说明: 下述命令使用Academic Torrents获取数据集。如果下载缓慢或失败，建议用户手动下载数据集并上传至Colab环境。
解压后，数据集通常包含三个子文件夹：benign/, malignant/, normal/，每个文件夹下有图像和对应的掩码文件。掩码文件命名为原图文件名加“_mask”后缀（若一个图像有多个肿块，则会有“_mask_2”等多份掩码）
stackoverflow.com
stackoverflow.com
。我们读取良性和恶性文件夹下的图像和掩码：

In [ ]:
# # 下载乳腺超声图像数据集 (约250MB)
# !wget -q -O BUSI.zip "https://academictorrents.com/download/d0b7b7ae40610bbeaea385aeb51658f527c86a16.torrent?torrent" || echo "Download started"
# !unzip -q BUSI.zip -d BUSI_Dataset
# !rm BUSI.zip


In [ ]:
import cv2
import os
import glob

ultrasound_images = []
ultrasound_masks = []

# 处理良性和恶性文件夹
for cls in ["benign", "malignant"]:
    image_paths = glob.glob(f"{BUSI_Dataset_dir}/{cls}/*.png")
    for img_path in image_paths:
        if "_mask" in img_path:
            continue  # 跳过掩码文件
        # 读取超声图像 (灰度PNG, cv2.imread默认读取为BGR三通道)
        img_bgr = cv2.imread(img_path)
        if img_bgr is None:
            continue
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        # 构造与图像尺寸相同的空白掩码
        mask_total = np.zeros(img_rgb.shape[:2], dtype=np.uint8)
        # 图像文件名基础部分（去掉路径和扩展名）
        base_name = os.path.splitext(img_path)[0]
        # 合并该图像的所有掩码文件（可能有多个肿块）
        mask_idx = 1
        while True:
            mask_path = f"{base_name}_mask.png" if mask_idx == 1 else f"{base_name}_mask_{mask_idx}.png"
            if os.path.exists(mask_path):
                mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
                if mask is not None:
                    mask_binary = (mask > 127).astype(np.uint8)
                    mask_total = np.logical_or(mask_total, mask_binary).astype(np.uint8)
                mask_idx += 1
            else:
                break
        # 如果该图像存在肿块标注，则保存
        if mask_total.sum() > 0:
            ultrasound_images.append(img_rgb.mean(axis=2).astype(np.float32))
            ultrasound_masks.append(mask_total)

print(f"乳腺超声图像总数: {len(ultrasound_images)} (良性+恶性), 掩码总数: {len(ultrasound_masks)}")


In [ ]:
dice_list_us = []
iou_list_us = []
hd_list_us = []

for idx, (slice_img, slice_mask) in enumerate(zip(ultrasound_images, ultrasound_masks)):
    # if idx > 20:
    #    break
    label2mask, label2box, pred_mask_total, true_mask, img_rgb = medsam_segment(slice_img, slice_mask, predictor)
    # combined_true_mask = (true_mask > 0)
    combined_true_mask = (true_mask == 1)

    # 计算评价指标
    dice = compute_dice(pred_mask_total, combined_true_mask)
    iou = compute_iou(pred_mask_total, combined_true_mask)
    hd = compute_hausdorff(pred_mask_total, combined_true_mask)
    dice_list_us.append(dice)
    iou_list_us.append(iou)
    hd_list_us.append(hd)
    # 打印处理进度
    print(f"\rProcessing ultrasound slice {idx+1}/{len(ultrasound_images)}: "
          f"Dice: {dice:.3f}, IoU: {iou:.3f}, HD: {hd:.1f}px", end='', flush=True)

In [ ]:
# 计算平均指标
dice_mean_us = np.mean(dice_list_us)
iou_mean_us = np.mean(iou_list_us)
hd_mean_us = np.mean([d for d in hd_list_us if math.isfinite(d)])
print(f"\n\n📊 Final Results Summary:")
print(f"   Total slices processed: {len(ultrasound_images)}")
print(f"   Slices with DICE < 0.5: {sum(1 for d in dice_list_us if d < 0.5)}")
print(f"Ultrasound 数据集: 平均Dice = {dice_mean_us:.4f}, 平均IoU = {iou_mean_us:.4f}, 平均Hausdorff距离 = {hd_mean_us:.2f} pixel")

In [ ]:
if fig is not None:
  fig.clear()
  plt.close(fig)
  fig = None
plt.ion()  # Turn on interactive mode
fig, axs = plt.subplots(3, 4)

idx = np.random.randint(0, len(ultrasound_images))
slice_img, slice_mask = ultrasound_images[idx], ultrasound_masks[idx]
label2mask, label2box, pred_mask_total, true_mask, img_rgb = medsam_segment(slice_img, slice_mask, predictor)
visualize_seg(fig, axs, label2box, label2mask, img_rgb, true_mask)

# Update figure title with metrics - highlight poor performance
dice, iou, hd = dice_list_us[idx], iou_list_us[idx], hd_list_us[idx]
fig.suptitle(f'Slice {idx+1}/{len(ultrasound_images)}\n'
            f'Dice: {dice:.3f}, IoU: {iou:.3f}, HD: {hd:.1f}px',
            fontsize=14, y=0.95, color='red')
# Clean up
plt.ioff()  # Turn off interactive mode
plt.tight_layout()
plt.show()
plt.close(fig)
fig = None